# Lineborn Sales LoRA
Fine-tunes Qwen3-4B-Instruct-2507 for consultative outbound sales. The frozen release benchmark is not used for training. Use a GPU runtime. This notebook now fails immediately if either training stage fails and refuses to download an incomplete adapter archive.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

def run(*args):
    print('>', ' '.join(map(str, args)), flush=True)
    return subprocess.run(list(map(str, args)), check=True)

run('nvidia-smi')
root = Path('/content/lineborn-runtime')
if root.exists():
    shutil.rmtree(root)
run('git', 'clone', '--depth', '1', '--branch', 'lineborn-sales-lora', 'https://github.com/SumamaAhmed69/Axemetric-Caller-Beta-Runtime.git', str(root))
os.chdir(root)
run(sys.executable, '-m', 'pip', 'install', '-q', '-r', 'training/requirements-colab.txt')
print('Environment ready:', root)

In [ ]:
# Build and validate the training corpus. Any validation failure stops the notebook.
run(sys.executable, 'training/build_sales_corpus.py')
run(sys.executable, 'training/validate_sales_corpus.py')

In [ ]:
# Stage 1: assistant-only QLoRA supervised fine-tuning.
run(sys.executable, 'training/train_sales_lora.py', '--max-length', '1536', '--epochs', '2', '--grad-accum', '16')
sft_model = Path('training/output/lineborn-sales-sft/adapter/adapter_model.safetensors')
assert sft_model.is_file() and sft_model.stat().st_size > 1024, 'SFT adapter was not produced'
print(f'SFT adapter ready: {sft_model.stat().st_size / 1024 / 1024:.1f} MiB')

In [ ]:
# Stage 2: preference optimization. This is required for the full candidate archive.
run(sys.executable, 'training/train_sales_dpo.py', '--max-length', '1536', '--epochs', '1', '--grad-accum', '16')
dpo_model = Path('training/output/lineborn-sales-dpo/adapter/adapter_model.safetensors')
assert dpo_model.is_file() and dpo_model.stat().st_size > 1024, 'DPO adapter was not produced'
print(f'DPO adapter ready: {dpo_model.stat().st_size / 1024 / 1024:.1f} MiB')

In [ ]:
# Verify every required adapter file, create a manifest with hashes, then package.
run(sys.executable, 'training/package_sales_adapters.py', '--require-dpo', '--archive', '/content/lineborn-sales-adapters.zip')
archive = Path('/content/lineborn-sales-adapters.zip')
assert archive.is_file() and archive.stat().st_size > 1024
print(f'Candidate archive ready: {archive.stat().st_size / 1024 / 1024:.1f} MiB')
from google.colab import files
files.download(str(archive))